# Visualizing XAI Top Nodes with System Graph

## Overview

With out GNN Models, we use the XAI library, Captum, to provide insights to the top nodes and features that our model attributed to model using Integrated Gradients (IG).

Using the attribution score that IG provides, we will determin the top 3 nodes and overlay that with out System Graph, giving the user insight to the parts of the system that is most effected.

## Load Dependencies

In [1]:
%%capture
%pip install neo4j orjson neo4j-viz neo4j-viz[neo4j] pandas

In [2]:
from neo4j_viz import Node, Relationship, VisualizationGraph
from neo4j import AsyncGraphDatabase
import pandas as pd

## Read XAI Results

In [3]:
dos_ranking = pd.read_csv("./data/dos_ranking.csv")
dos2_ranking = pd.read_csv("./data/dos2_ranking.csv")
mitm_ranking = pd.read_csv("./data/mitm_ranking.csv")
mitm2_ranking = pd.read_csv("./data/mitm2_ranking.csv")
physical_ranking = pd.read_csv("./data/physical_ranking.csv")
physical2_ranking = pd.read_csv("./data/physical2_ranking.csv")

In [53]:
top_nodes = {
    "dos": (dos_ranking['original_id'][:3],dos_ranking['snapshot_id'][:3]),
    "dos2": (dos2_ranking['original_id'][:3], dos2_ranking['snapshot_id'][:3]),
    "mitm": (mitm_ranking['original_id'][:3], mitm_ranking['snapshot_id'][:3]),
    "mitm2": (mitm2_ranking['original_id'][:3], mitm2_ranking['snapshot_id'][:3]),
    "physical": (physical_ranking['original_id'][:3], physical_ranking['snapshot_id'][:3]),
    "physical2": (physical2_ranking['original_id'][:3], physical2_ranking['snapshot_id'][:3])
}


## Load System Graph + XAI Nodes From Neo4j

In [76]:
from neo4j import GraphDatabase, RoutingControl, Result
from neo4j_viz.neo4j import from_neo4j

query = """
MATCH (n)-[r]-(a) WHERE type(r) in ['FEEDS_TO','FEEDS_THROUGH','CONTROLS','ISSUE_COMMANDS', 'READS_FROM','SENSOR_ON','HAS_ENDPOINT'] AND n.asset_type <> 'External'
OPTIONAL MATCH (e)<-[:HAS_ENDPOINT]-(n) 
OPTIONAL MATCH (t1:Connection)-[r1]-(n1) WHERE id(t1) = $t1_id AND type(r1) <> 'CONTAINS'
OPTIONAL MATCH (t2:Connection)-[r2]-(n2) WHERE id(t2) = $t2_id AND type(r2) <> 'CONTAINS'
OPTIONAL MATCH (t3:Connection)-[r3]-(n3) WHERE id(t3) = $t3_id AND type(r3) <> 'CONTAINS'
return n,r,e, t1, t2, t3, r1, n1, r2, n2, r3, n3
"""
graphs = {}

# get key and value from top_nodes
with GraphDatabase.driver(
            'bolt://localhost:7698',
        auth=('neo4j', 'changeme')
    ) as neo4j_instance:
    for key, val in top_nodes.items():    
        with neo4j_instance.session() as session:
            VG = from_neo4j(session.run(query,t1_id=val[0][0],t2_id=val[0][1], t3_id=val[0][2], snap_id=val[1]))
            VG.color_nodes(field="Caption")
            VG.set_node_captions(property="asset_name")            
            
            for node in VG.nodes:
                print(f"Processing node {node.id}")
                print(f"anomaly ids: {val[0].values}")
                if node.caption is None or node.caption == "":
                    cap = node.properties.get("ip", None)
                    if cap is None:
                        cap = node.properties.get("mac", None)
                    if cap is None:
                        #default to label
                        cap = node.properties.get("labels", "Unknown")
                        cap = ",".join(cap) if isinstance(cap, list) else str(cap)
                    node.caption = cap
                if int(node.id) in val[0].values:
                    print(f"Highlighting node {node.properties.get('id')}")
                    node.properties["highlight"] = 2
                # if network node with label of HMI, PLC, or Endpoint, highlight in blue
                elif "HMI" in node.properties.get("labels", []) or "PLC" in node.properties.get("labels", []) or "Endpoint" in node.properties.get("labels", []):
                    print(f"Highlighting node {node.properties.get('id')} as network device")
                    node.properties["highlight"] = 3
                else:
                    node.properties["highlight"] = 1
            VG.color_nodes(property="highlight", colors=["#00FF00", "#FF0000"])
            html = VG.render(height="100%")
            with open(f"../../exports/images/anomaly_graphs_{key}.html", "w", encoding="utf-8") as f:
                f.write(html.data)

            
  

Processing node 0
anomaly ids: [   636    638 263005]
Processing node 38
anomaly ids: [   636    638 263005]
Processing node 263005
anomaly ids: [   636    638 263005]
Highlighting node None
Processing node 59
anomaly ids: [   636    638 263005]
Highlighting node None as network device
Processing node 60
anomaly ids: [   636    638 263005]
Highlighting node None as network device
Processing node 52
anomaly ids: [   636    638 263005]
Processing node 34
anomaly ids: [   636    638 263005]
Processing node 40
anomaly ids: [   636    638 263005]
Processing node 36
anomaly ids: [   636    638 263005]
Processing node 32
anomaly ids: [   636    638 263005]
Processing node 41
anomaly ids: [   636    638 263005]
Processing node 39
anomaly ids: [   636    638 263005]
Processing node 35
anomaly ids: [   636    638 263005]
Processing node 37
anomaly ids: [   636    638 263005]
Processing node 33
anomaly ids: [   636    638 263005]
Processing node 48
anomaly ids: [   636    638 263005]
Processing n

c:\Projects\CIS\.venv\Lib\site-packages\neo4j_viz\visualization_graph.py:512: UserWarning: Ran out of colors for property 'highlight'. 3 colors were needed, but only 2 were given, so reused colors
  warnings.warn(


Processing node 0
anomaly ids: [   636    640 197609]
Processing node 38
anomaly ids: [   636    640 197609]
Processing node 197609
anomaly ids: [   636    640 197609]
Highlighting node None
Processing node 57
anomaly ids: [   636    640 197609]
Highlighting node None as network device
Processing node 60
anomaly ids: [   636    640 197609]
Highlighting node None as network device
Processing node 52
anomaly ids: [   636    640 197609]
Processing node 34
anomaly ids: [   636    640 197609]
Processing node 40
anomaly ids: [   636    640 197609]
Processing node 36
anomaly ids: [   636    640 197609]
Processing node 32
anomaly ids: [   636    640 197609]
Processing node 41
anomaly ids: [   636    640 197609]
Processing node 39
anomaly ids: [   636    640 197609]
Processing node 35
anomaly ids: [   636    640 197609]
Processing node 37
anomaly ids: [   636    640 197609]
Processing node 33
anomaly ids: [   636    640 197609]
Processing node 48
anomaly ids: [   636    640 197609]
Processing n